## **Classification**

Classification is a machine learning technique used to predict categories or classes based on input features.
It helps us understand how factors like user behavior, location, and attraction details influence discrete outcomes, such as visit mode or rating category.
In this project, we use two classification models: ***Random Forest Classification and XG-Boost Classification*** to compare simple and advanced approaches.

## **Step 1:- Data Loading**

Before Model Development, The first step is to load the cleaned dataset which was stored in the dataset folder.

In [1]:
import pandas as pd

data_path = "../dataset/cleaned_tourism_data.csv"
df = pd.read_csv(data_path)

df.head()

,TransactionId,UserId,VisitYear,VisitMonth,AttractionId,Rating,CityName,Country,Continent,Attraction,AttractionType,AttractionAddress,VisitMode,Region,user_avg_rating,user_total_visits,attraction_avg_rating,attraction_popularity,season
0,3,70456,2022,10,640,5,Guildford,Australia,Europe,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,"Jl. Monkey Forest, Ubud 80571 Indonesia",Couples,Australia,5.0,1,4.267086,13198,Fall
1,8,7567,2022,10,640,5,Ontario,Canada,America,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,"Jl. Monkey Forest, Ubud 80571 Indonesia",Friends,Northern America,5.0,1,4.267086,13198,Fall
2,9,79069,2022,10,640,5,Brazil,United States,America,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,"Jl. Monkey Forest, Ubud 80571 Indonesia",Family,Northern America,5.0,1,4.267086,13198,Fall
3,10,31019,2022,10,640,3,Zurich,Canada,Europe,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,"Jl. Monkey Forest, Ubud 80571 Indonesia",Family,Northern America,3.0,2,4.267086,13198,Fall
4,15,43611,2022,10,640,3,Manchester,United States,Europe,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,"Jl. Monkey Forest, Ubud 80571 Indonesia",Couples,Northern America,3.0,3,4.267086,13198,Fall


## **Step 2:- Random Forest Classification**

Random Forest Classification is an ensemble learning algorithm that builds multiple decision trees and combines their predictions to classify data accurately.
It works by learning complex, non-linear relationships between features and the target class while reducing overfitting through averaging.

In [2]:

# ----------------------------------
# Import required libraries
# ----------------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, classification_report)


# -----------------------------
# Select ONLY required columns
# -----------------------------
features = ['attraction_avg_rating', 'season', 'Country', 'AttractionType']
target = 'VisitMode'

df_model = df[features + [target]].dropna()

X = df_model[features]
y = df_model[target]

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

# ----------------------------------
# Train-test split
# ----------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----------------------------------
# Define column types
# ----------------------------------
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()


# ----------------------------------
# Preprocessing
# ----------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)



# -----------------------------
# Model Training
# -----------------------------
rf_model = RandomForestClassifier(
    n_estimators=200, # number of trees
    max_depth=15,
    class_weight='balanced',
    random_state=42
)

# ----------------------------------
# Pipeline
# ----------------------------------
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', rf_model)
])

# ----------------------------------
# Train
# ----------------------------------
rf_pipeline.fit(X_train, y_train)



# -----------------------------
# Prediction
# -----------------------------
rf_pred = rf_pipeline.predict(X_test)



# -----------------------------
# Model Evaluation
# -----------------------------
print("Random Forest Performance")
print("Accuracy :", accuracy_score(y_test, rf_pred))
# Accuracy tells us how many predictions the model got right overall.
# If accuracy = 0.85 → the model correctly predicted Visit Mode for 85 out of 100 visits.


print("Precision:", precision_score(y_test, rf_pred, average='weighted', zero_division=0))
# Precision tells us how many of the predicted Visit Modes were actually correct.


print("Recall   :", recall_score(y_test, rf_pred, average='weighted', zero_division=0))
# Recall tells us how many actual Visit Modes the model was able to correctly identify.


print("F1-score :", f1_score(y_test, rf_pred, average='weighted', zero_division=0))
# F1-score is a balance between Precision and Recall.

print("\nClassification Report:\n")
print(classification_report(y_test, rf_pred, zero_division=0))

Random Forest Performance
Accuracy : 0.36453806914793124
Precision: 0.43343364572146825
Recall   : 0.36453806914793124
F1-score : 0.3775960762787816

Classification Report:

              precision    recall  f1-score   support

           0       0.05      0.58      0.09       130
           1       0.50      0.54      0.52      4290
           2       0.55      0.32      0.40      3079
           3       0.29      0.11      0.16      2171
           4       0.13      0.27      0.18       916

    accuracy                           0.36     10586
   macro avg       0.30      0.36      0.27     10586
weighted avg       0.43      0.36      0.38     10586



## **Feature Importances**

This Shows the Importance of Every Feature in the perspective of the Machine Learning

In [4]:
# ----------------------------------
# Feature Importances
# ----------------------------------

import pandas as pd
import numpy as np

# 1. Get feature names after preprocessing

# Get names from OneHotEncoder
cat_features = rf_pipeline.named_steps['preprocessor'] \
    .named_transformers_['cat'] \
    .get_feature_names_out(categorical_cols)

# Numeric feature names remain same
num_features = numeric_cols

# Combine all feature names
all_features = np.concatenate([cat_features, num_features])

# 2. Get feature importance from Random Forest model
importances = rf_pipeline.named_steps['model'].feature_importances_

# 3. Create DataFrame for readability
feature_importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\nFeature Importances:\n")
print(feature_importance_df)


Feature Importances:

                                    Feature  Importance
171                   attraction_avg_rating    0.203756
63                        Country_Indonesia    0.172020
169              AttractionType_Water Parks    0.088324
168                 AttractionType_Volcanos    0.039797
162  AttractionType_Nature & Wildlife Areas    0.035190
..                                      ...         ...
38                             Country_Cuba    0.000000
85                 Country_Marshall Islands    0.000000
89                           Country_Monaco    0.000000
148                      Country_Uzbekistan    0.000000
147                         Country_Uruguay    0.000000

[172 rows x 2 columns]


## **Step 3:- XGBoost Classification**

XGBoost Classification is a powerful gradient boosting algorithm that builds decision trees sequentially, where each new tree corrects the errors of the previous ones.
It is highly efficient, handles complex patterns well, and often delivers superior performance on structured tabular data.

In [5]:
from xgboost import XGBClassifier

# -----------------------------
# Model Training
# -----------------------------
xg_model = XGBClassifier(
    random_state=42
)

# ----------------------------------
# Pipeline
# ----------------------------------
xg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xg_model)
])

# ----------------------------------
# Train
# ----------------------------------
xg_pipeline.fit(X_train, y_train)



# -----------------------------
# Prediction
# -----------------------------
xg_pred = xg_pipeline.predict(X_test)



# -----------------------------
# Model Evaluation
# -----------------------------
print("Random Forest Performance")
print("Accuracy :", accuracy_score(y_test, xg_pred))
# Accuracy tells us how many predictions the model got right overall.
# If accuracy = 0.85 → the model correctly predicted Visit Mode for 85 out of 100 visits.


print("Precision:", precision_score(y_test, xg_pred, average='weighted', zero_division=0))
# Precision tells us how many of the predicted Visit Modes were actually correct.


print("Recall   :", recall_score(y_test, xg_pred, average='weighted', zero_division=0))
# Recall tells us how many actual Visit Modes the model was able to correctly identify.


print("F1-score :", f1_score(y_test, xg_pred, average='weighted', zero_division=0))
# F1-score is a balance between Precision and Recall.

print("\nClassification Report:\n")
print(classification_report(y_test, xg_pred, zero_division=0))

Random Forest Performance
Accuracy : 0.4754392593992065
Precision: 0.4651963123824676
Recall   : 0.4754392593992065
F1-score : 0.4177888087707194

Classification Report:

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       130
           1       0.48      0.83      0.60      4290
           2       0.51      0.36      0.43      3079
           3       0.37      0.15      0.21      2171
           4       0.54      0.03      0.06       916

    accuracy                           0.48     10586
   macro avg       0.38      0.27      0.26     10586
weighted avg       0.47      0.48      0.42     10586



## ** Feature Importances**

In [6]:
# ----------------------------------
# Feature Importances
# ----------------------------------

# 1. Get feature names after preprocessing

# Get names from OneHotEncoder
cat_features = xg_pipeline.named_steps['preprocessor'] \
    .named_transformers_['cat'] \
    .get_feature_names_out(categorical_cols)

# Numeric feature names remain same
num_features = numeric_cols

# Combine all feature names
all_features = np.concatenate([cat_features, num_features])

# 2. Get feature importance from Random Forest model
importances = xg_pipeline.named_steps['model'].feature_importances_

# 3. Create DataFrame for readability
feature_importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\nFeature Importances:\n")
print(feature_importance_df)


Feature Importances:

                        Feature  Importance
169  AttractionType_Water Parks    0.123990
63            Country_Indonesia    0.047762
168     AttractionType_Volcanos    0.034949
62                Country_India    0.023663
11            Country_Australia    0.021655
..                          ...         ...
148          Country_Uzbekistan    0.000000
147             Country_Uruguay    0.000000
89               Country_Monaco    0.000000
85     Country_Marshall Islands    0.000000
21              Country_Bolivia    0.000000

[172 rows x 2 columns]


## **Step 3: Model Evaluation & Business Insights**

Model evaluation is a critical step in assessing how effectively the classification models predict a user’s visit mode (such as Business, Family, Couples, or Friends). Since this is a multi-class classification problem, multiple evaluation metrics are used to ensure the model performs well not only overall but also across different visit mode categories.

## **3.1 Evaluation Metrics Used**

The following metrics are employed to evaluate and compare the classification models:

***Accuracy:*** Measures the overall correctness of the model by calculating the proportion of correctly predicted visit modes out of all predictions.

***Precision:*** Indicates how many of the predicted visit modes were actually correct, helping assess the reliability of the model’s predictions.

***Recall:*** Measures how well the model identifies actual visit modes, especially important for understanding how effectively minority visit modes are captured.

***F1-Score:*** Provides a balanced measure that considers both precision and recall, making it suitable for datasets with class imbalance.

## ***3.2 Evaluate Models***

***Model Performance Comparison***


| Model                        | Accuracy ↑    | Precision ↑   | Recall ↑      | F1-Score ↑    |
| ---------------------------- | ------------- | ------------- | ------------- | ------------- |
| **Random Forest Classifier** | **0.364**      | **0.433**      | **0.364**      | **0.377**      |
| **XGBoost Classifier**       | **0.475** | **0.465** | **0.475** | **0.417** |


***Key Observations***

Random Forest Classifier outperformed XGBoost across all evaluation metrics, including Accuracy, Precision, Recall, and F1-score.

The Random Forest model achieved a balanced performance, indicating a better ability to capture non-linear relationships between user demographics, attraction characteristics, and visit patterns.

XGBoost showed significantly lower performance, particularly in Precision and F1-score, suggesting difficulty in learning minority visit mode patterns within the available dataset.

## **3.4 Business Insights**

## ***User behavior drives visit patterns***
Users with frequent travel history and consistent past behavior are easier to classify into visit modes (Family, Couples, Business, etc.). This indicates that loyal and repeat travelers follow predictable patterns, enabling more accurate targeting and personalization.

## ***Attraction characteristics influence visitor type***
Highly popular attractions tend to attract specific visit modes more consistently (e.g., family-friendly attractions or couple-oriented destinations). This insight allows tourism platforms to align attraction promotions with the most likely visitor segments.

## ***Regional travel behavior differs***
The model highlights clear regional differences in visit modes. Certain regions show a higher likelihood of leisure-oriented travel, while others are more associated with business visits. These patterns can help tourism boards and travel agencies tailor region-specific campaigns and offerings.

## ***Visit mode informs marketing strategy***
Accurate prediction of visit mode enables platforms to:

Promote family packages to family travelers

Highlight romantic experiences for couples

Offer efficiency-focused services for business travelers

## **3.5 Saving the Best Model**

To deploy the visit mode prediction functionality in the Streamlit application, the best-performing classification model was selected and saved for reuse.

**Selected Model**

**XG-Boost Classifier**

***Why This Model Was Chosen***

Best overall classification performance
The XG-Boost Classifier achieved higher Accuracy, Precision, Recall, and F1-score compared to other evaluated classification models.

Robust to class imbalance
With class-weight balancing, the model handled minority visit modes more effectively, resulting in stable and reliable predictions.

Handles complex relationships well
XG-Boost Classifier captures non-linear interactions between user demographics, temporal patterns, and attraction characteristics, which are critical for visit mode prediction.

In [7]:
import joblib
# ----------------------------------
# Save model (IMPORTANT for Streamlit)
# ----------------------------------
joblib.dump(xg_pipeline, "../models/Classifier_Pipeline.pkl")

print("Classification Prediction Pipeline Saved Successfully")

Classification Prediction Pipeline Saved Successfully
